In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

## Ingestion del carpeta "production_company"

###Paso 1 - Leer los archivos CSV usando "DataframeReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
productions_companies_schema = StructType([
    StructField("companyId", IntegerType(), True),
    StructField("companyName", StringType(), True)
])

In [0]:
productions_companies_df = spark.read\
    .schema(productions_companies_schema)\
    .csv(f"{bronze_folder_path}/{v_file_date}/production_company")

### Paso 2 - Renombrar las columnas y añadir nuevas columnas
1. "companyId" renombrar a "company_id"
2. "companyName" renombrar a "company_name"
3. Agregar las columnas "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
productions_companies_final_df = add_ingestion_date(productions_companies_df)\
    .withColumnsRenamed({"companyId": "company_id",
                         "companyName": "company_name"})\
    .withColumn("enviroments", lit("Production"))\
    .withColumn("file_date", lit(v_file_date))

### Paso 3 - Escribir la salida en un formato "Parquet" PartitionBy

In [0]:
#overwrite_partition("movie_silver", "productions_companies", "file_date", v_file_date)

In [0]:
#productions_companies_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/productions_companies")

In [0]:
#productions_companies_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.productions_companies")

condition_merge = 'tgt.company_id = src.company_id AND tgt.file_date = src.file_date'

incremental_merge("movie_silver", "productions_companies", productions_companies_final_df, condition_merge, "file_date")


In [0]:
%sql
SELECT file_date, count(1)
FROM movie_silver.productions_companies
GROUP BY file_date;

file_date,count(1)
2024-12-16,3000
2024-12-23,1000
2024-12-30,1047


In [0]:
dbutils.notebook.exit("Exitoso")